In [2]:
import pandas as pd
import os

In [92]:
# Load all election results into a dictionary mapping knesset number to the corresponding DataFrame
knesset_no_to_parties_passed_threshold = {}
files = os.listdir('../data/')
for file in files:
    if file.startswith('elections') and file.endswith('.csv'):
        knesset_no = int(file.split('_')[1].split('.')[0])
        knesset_no_to_parties_passed_threshold[knesset_no] = pd.read_csv(os.path.join('../data/', file))

In [93]:
# Some of the columns have מרץ instead of מרצ, so we need to normalize the party signs
for knesset_no, df in knesset_no_to_parties_passed_threshold.items():
    if "מרץ" in df.columns:
        df.rename(columns={"מרץ": "מרצ"}, inplace=True)

In [94]:
df_elections_results_all_parties = pd.read_csv("../data/knesset_election_results_18_to_25.csv")

In [95]:
# Check that all DataFrames contain the party columns that passed the threshold.
missing_columns = {}
for knesset_no, df in knesset_no_to_parties_passed_threshold.items():
    parties_sign = (
        df_elections_results_all_parties
        .query(f"knesset_number == {knesset_no}")
        ["party_sign"]
        .to_list()
    )
    for col in parties_sign:
        if col not in df.columns:
            missing_columns[knesset_no] = missing_columns.get(knesset_no, []) + [col]

if missing_columns:
    print("Missing columns:")
    for knesset_no, cols in missing_columns.items():
        print(f"Knesset {knesset_no}: {cols}")
else:
    print("No missing columns found.")

No missing columns found.


In [ ]:
for knesset_no, df in knesset_no_to_parties_passed_threshold.items():
    display(knesset_no)
    display(df[df.columns[0:10]].head(1))

needed_columns = {
    18: ["סמל ישוב", "סמל קלפי"],
    19: ["סמל ישוב", "מספר קלפי"],
    20: ["סמל ישוב", "מספר קלפי"],
    21: ["סמל ישוב", "מספר קלפי"],
    22: ["סמל ישוב", "קלפי"],
    23: ["סמל ישוב", "קלפי"],
    24: ["סמל ישוב", "קלפי"],
    25: ["סמל ישוב", "קלפי"],
}



In [97]:

# Check that all DataFrames contain the needed columns for the corresponding knesset number
for knesset_no, df in knesset_no_to_parties_passed_threshold.items():
    if knesset_no in needed_columns:
        for col in needed_columns[knesset_no]:
            if col not in df.columns:
                print(f"Knesset {knesset_no} is missing column: {col}")

In [99]:
def normalize_knesset_results(
    knesset_no: int,
    df_elections_results_all_parties: pd.DataFrame,
    needed_columns: dict[int, list[str]],
    knesset_no_to_parties_passed_threshold: dict[int, pd.DataFrame]
) -> pd.DataFrame:
    parties_columns = df_elections_results_all_parties.query(f"knesset_number == {knesset_no}")["party_sign"].to_list()
    base_columns = needed_columns[knesset_no]
    all_columns = base_columns + parties_columns
    df_knesset_18 = knesset_no_to_parties_passed_threshold[knesset_no][all_columns]
    return (
        df_knesset_18.melt(
            id_vars=base_columns,
            var_name="party_sign",
            value_name="votes"
        )
        .assign(knesset_number=knesset_no)
        .rename(columns={base_columns[0]: "locality_id", base_columns[1]: "kalpi_id"})
    )


In [100]:
all_dfs = []
for knesset_no in range(18, 26):
    df_knesset = normalize_knesset_results(
        knesset_no=knesset_no,
        df_elections_results_all_parties=df_elections_results_all_parties,
        needed_columns=needed_columns,
        knesset_no_to_parties_passed_threshold=knesset_no_to_parties_passed_threshold
    )
    all_dfs.append(df_knesset)

In [115]:
df_full = pd.concat(all_dfs)


In [119]:
df_full.to_csv("../data/normalized_election_results_18_to_25.csv", index=False, encoding="utf-8-sig")

In [4]:
# Check rows that can be deleted (rows with 0 votes for all parties)
df_full = pd.read_csv("../data/normalized_election_results_18_to_25.csv", encoding="utf-8-sig")


In [5]:
# (
#     df_full
#     .groupby(["locality_id", "kalpi_id", "knesset_number"])["votes"]
#     .sum().reset_index()
#     .query("votes > 0")
# )
display(f"Total records: {len(df_full)}")
display(f"Records with 0 votes: {len(df_full[df_full['votes'] == 0])}")

df_full[
    df_full.groupby(["locality_id", "kalpi_id", "knesset_number"])["votes"].transform("sum") != 0
].copy()

'Total records: 936060'

'Records with 0 votes: 184457'

,locality_id,kalpi_id,party_sign,votes,knesset_number
0,967,1.0,כן,0,18
1,967,2.0,כן,1,18
2,472,1.0,כן,24,18
3,472,2.0,כן,20,18
4,472,3.0,כן,47,18
...,...,...,...,...,...
936055,3563,3.0,אמת,4,25
936056,3563,4.0,אמת,6,25
936057,970,1.0,אמת,0,25
936058,1346,1.0,אמת,1,25


In [13]:
# Remove kalpi that don't have address
# df_kalpi_address = 

df_kalpi_address = (
    pd
    .read_csv("../data/kalpi_address_with_coords.csv", encoding="utf-8-sig")
    .query("coordinates.notna()")
)

In [15]:
df_full.columns

Index(['locality_id', 'kalpi_id', 'party_sign', 'votes', 'knesset_number'], dtype='str')

In [20]:
(
    pd.merge(
        df_full,
        df_kalpi_address,
        on=["locality_id", "kalpi_id"],
        how="inner"
    )
    .query("votes > 0")
    [['locality_id', 'kalpi_id', 'party_sign', 'votes', 'knesset_number']]
    .to_csv("../data/normalized_election_results_18_to_25.csv", index=False, encoding="utf-8-sig")
)

/tmp/ipykernel_1028/2983343271.py:2: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  pd.merge(
